In [5]:
# Imports
import sys
import warnings
warnings.filterwarnings("ignore")
sys.path.append("/projeto")

from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from conf.spark_session import get_spark_session
from delta.tables import DeltaTable
from datetime import datetime

# Sessão do Spark com Delta Lake
spark = get_spark_session()
spark.sparkContext.setLogLevel("ERROR")

In [11]:
# Lendo a tabela de controle de execução do Hive com Spark
df_controle_execucao = spark.table("lakehouse.controle_execucao")
df_controle_execucao.show(5,False)

+---------------+----------------+----------+--------------+--------------------------+-----------+-------+-------------+---------+-------------------+--------------------------+
|execution_id   |pipeline        |source    |ingestion_date|execution_timestamp       |total_files|status |error_message|processed|processed_timestamp|created_at                |
+---------------+----------------+----------+--------------+--------------------------+-----------+-------+-------------+---------+-------------------+--------------------------+
|20260310T211929|github_ingestion|github_api|2026-03-10    |2026-03-10T21:24:13.641205|1000       |success|NULL         |false    |NULL               |2026-03-10 21:25:49.741468|
+---------------+----------------+----------+--------------+--------------------------+-----------+-------+-------------+---------+-------------------+--------------------------+



In [12]:
# Buscar a execução pendente na tabela de controle de execução
df_execucao = spark.sql("""
    SELECT execution_id
    FROM lakehouse.controle_execucao
    WHERE processed = false
    ORDER BY created_at
    LIMIT 1
""")

df_execucao.createOrReplaceTempView("df_execucao")
df_execucao.show()

+---------------+
|   execution_id|
+---------------+
|20260310T211929|
+---------------+



In [13]:
# Salvando o id de execução pendende em uma variavel
execution_id = df_execucao.collect()[0]["execution_id"]
print(id_execucao)

# Definindo a data de ingestão
ingestion_date = datetime.strptime(execution_id[:8], "%Y%m%d").strftime("%Y-%m-%d")
print(ingestion_date)

20260310T211929
2026-03-10


In [14]:
# Montando o path para ingestão
raw_path = f"""
s3a://datalake/bronze/raw/
ingestion_date={ingestion_date}/
execution_id={execution_id}
""".replace("\n","")

print(raw_path)

s3a://datalake/bronze/raw/ingestion_date=2026-03-10/execution_id=20260310T211929


In [15]:
# Ler os arquivos da última execução
df_raw = (
    spark.read
    .option("multiline", True)
    .json(raw_path)
)

In [17]:
# Adicionando colunas para controle de ingestão
df_bronze = (
    df_raw
    .withColumn("ingestion_date", F.lit(ingestion_date))
    .withColumn("execution_id", F.lit(execution_id))
    .withColumn("created_at", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
)

df_bronze.limit(5).show()

+--------------------+------------+-----------+--------------+---------------+--------------------+--------------------+
|               entry|resourceType|       type|ingestion_date|   execution_id|          created_at|         source_file|
+--------------------+------------+-----------+--------------+---------------+--------------------+--------------------+
|[{urn:uuid:071812...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 00:54:...|s3a://datalake/br...|
|[{urn:uuid:f156b8...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 00:54:...|s3a://datalake/br...|
|[{urn:uuid:b0f49c...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 00:54:...|s3a://datalake/br...|
|[{urn:uuid:37ff59...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 00:54:...|s3a://datalake/br...|
|[{urn:uuid:cbaf79...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 00:54:...|s3a://datalake/br...|
+--------------------+----------

In [18]:
# Verificando o schema do dataframe
df_bronze.printSchema()

root
 |-- entry: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- fullUrl: string (nullable = true)
 |    |    |-- request: struct (nullable = true)
 |    |    |    |-- method: string (nullable = true)
 |    |    |    |-- url: string (nullable = true)
 |    |    |-- resource: struct (nullable = true)
 |    |    |    |-- abatementDateTime: string (nullable = true)
 |    |    |    |-- achievementStatus: struct (nullable = true)
 |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |-- active: boolean (nullable = true)
 |    |    |    |-- activity: array (nullable = true)
 |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |    |    |-- code

In [19]:
# Contagem de registros
df_bronze.count()

1000

In [20]:
# Definindo o path na camada bronze
bronze_path = "s3a://datalake/bronze/pacientes"

(
    df_bronze.write
    .format("delta")
    .mode("append")
    .partitionBy("ingestion_date")
    .option("mergeSchema", "true")
    .save(bronze_path)
)

In [21]:
df_pacientes = spark.read.format("delta").load(bronze_path)
df_pacientes.limit(5).show()

+--------------------+------------+-----------+--------------+---------------+--------------------+--------------------+
|               entry|resourceType|       type|ingestion_date|   execution_id|          created_at|         source_file|
+--------------------+------------+-----------+--------------+---------------+--------------------+--------------------+
|[{urn:uuid:24c0e4...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 01:02:...|s3a://datalake/br...|
|[{urn:uuid:597f10...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 01:02:...|s3a://datalake/br...|
|[{urn:uuid:2aba1a...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 01:02:...|s3a://datalake/br...|
|[{urn:uuid:728a43...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 01:02:...|s3a://datalake/br...|
|[{urn:uuid:4702b5...|      Bundle|transaction|    2026-03-10|20260310T211929|2026-03-12 01:02:...|s3a://datalake/br...|
+--------------------+----------

In [22]:
# Atualiuzando a tabela de controle de execução
spark.sql(f"""
    UPDATE lakehouse.controle_execucao
    SET
        processed = true,
        processed_timestamp = current_timestamp()
    WHERE execution_id = '{execution_id}'
""")

DataFrame[num_affected_rows: bigint]

In [23]:
# Lendo a tabela controle de execução do Hive com Spark
df = spark.table("lakehouse.controle_execucao")
df.show()

+---------------+----------------+----------+--------------+--------------------+-----------+-------+-------------+---------+--------------------+--------------------+
|   execution_id|        pipeline|    source|ingestion_date| execution_timestamp|total_files| status|error_message|processed| processed_timestamp|          created_at|
+---------------+----------------+----------+--------------+--------------------+-----------+-------+-------------+---------+--------------------+--------------------+
|20260310T211929|github_ingestion|github_api|    2026-03-10|2026-03-10T21:24:...|       1000|success|         NULL|     true|2026-03-12 01:07:...|2026-03-10 21:25:...|
+---------------+----------------+----------+--------------+--------------------+-----------+-------+-------------+---------+--------------------+--------------------+

